# 20c · Walk-forward with a looser entry gate, plus an ungated control (E20, part 3)

In 20b the best rule on the history was always a top/bottom **1 %** signal (f5 or f6). Those readings were so rare
that the rule and the gated agent made no trade at all in T4 and V2, and only 1 trade in T3 and V1, so the
agent was never really tested. 20c repeats 20b exactly, with two changes:

1. **Looser gate.** The rules (benchmark and gate) may only use the **top/bottom 10 % or 5 %** thresholds
   (quantiles 0.90 and 0.95 of |signal| on the history), so signals fire often enough to be judged. The best rule
   per fold is still chosen on everything before the judged window.
2. **Ungated control.** In every fold the same Double DQN is also trained with no entry gate, to show whether the
   gate itself helps.

| fold | agent trains on | agent checkpoint chosen on | rule chosen on | judged on |
|---|---|---|---|---|
| 1 | T1 | T2 | T1-T2 | T3 |
| 2 | T1-T2 | T3 | T1-T3 | T4 |
| 3 | T1-T3 | T4 | T1-T4 | V1 |
| 4 | T1-T4 | V1 | T1-T4 + V1 | V2 |

Caveats: (a) the looser gate was chosen **after** seeing 20b's results on the judged windows, so T3, T4, V1 and V2
are not fully fresh for this choice; the untouched test and stress_bear periods remain the final check. (b) E19 and
20a found the edge per trade clearly beats the 0.24 % round-trip cost only at the 1 % extremes; at 5-10 % it is
smaller, so more trades may mean more losses unless the agent filters them. (c) As in 20b: T3/T4 factors come from
models that also saw later train blocks, and validation already chose the E15 GRU and the E16 hyperparameters.

**Run 20a first** (20b is not needed). T4 GPU. Rules take minutes; agents about 20-25 minutes per run (4 folds x 3
seeds x 2 variants x 1M steps, about 9 hours). Runs resume after a disconnect. Writes
`thesis_rl/initiation/e20c_walkforward/` and `report/e20c_*`. **Test and stress_bear are not loaded.**

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "20c_walkforward_loose_gate")

In [ ]:
import json, time, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from rltrade import factor_rule as FR, walkforward as WF, results as R
from rltrade.env import EnvConfig, Market, run_policy
from rltrade.baselines import buy_and_hold_policy
from rltrade.metrics import compute_metrics
from rltrade.runner import RunSpec, run, backtest_saved

INIT = L.root / "initiation"
OUT, REP = INIT / "e20c_walkforward", INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"
NEEDED = [f"fixed_plus/market_{s}.npz" for s in ("train", "val")] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)   # test / stress_bear stay zipped
    print("unzipped", NEEDED, "->", LOCAL)
F = INIT / "e20" / "crossfit_mean3"
assert (F / "crossfit.json").exists(), "run notebook 20a first (cross-fitted factors)"
N = {s: len(np.load(PREP / f"market_{s}.npz")["midpoint"]) for s in ("train", "val")}
for s in N:
    assert np.load(F / f"factors_{s}.npy", mmap_mode="r").shape == (N[s], 8), s
WINS = WF.windows(N["train"], N["val"])
FOLD_DIRS = [WF.write_fold(PREP, F, WINS, f, LOCAL / "e20_folds" / f"fold{f + 1}") for f in range(len(WF.FOLDS))]

hp = json.loads((INIT / "agents" / "tuning" / "best_hp.json").read_text())       # E16's tuned Double DQN
AGENT_HP, ENV_OVER = hp["agent_hp"], hp["env"]
AGENT_HP["net_dims"] = tuple(AGENT_HP["net_dims"])
NET = EnvConfig(state="A", **ENV_OVER)
PPY = NET.periods_per_year
SEEDS, TOTAL_STEPS, EVAL_EVERY, MIN_TRADES = (0, 1, 2), 1_000_000, 100_000, 5
QUANTILES = (0.90, 0.95)             # gate / rule thresholds: top-bottom 10 % or 5 % of |signal| on the history (20b: 0.5 ... 0.99)
RUN_UNGATED = True                   # also train the plain E16-style agent per fold (a control; doubles the time)
times = {w: R.as_datetime(np.load(PREP / f"market_{s}.npz")["time_ns"][[lo, hi - 1]]) for w, (s, lo, hi) in WINS.items()}
for w, (s, lo, hi) in WINS.items():
    print(f"{w}: {s} rows {lo:,}-{hi:,}  {times[w][0]:%Y-%m-%d} -> {times[w][1]:%Y-%m-%d}")
print("agent:", AGENT_HP, ENV_OVER)

## 1 · Benchmark: the best rule on the history, traded on the next window

In [ ]:
MK = lambda f, s, cfg=NET: Market(FOLD_DIRS[f], FOLD_DIRS[f], s, cfg, "cpu")       # noqa: E731
rule_rows, RULES, BT_RULE, BT_BH, all_rules = [], [], [], [], []
for f, (tr_w, sel_w, ev_w) in enumerate(WF.FOLDS):
    sig_hist = FR.signals(np.load(FOLD_DIRS[f] / "factors_hist.npy"))
    grid = FR.make_grid(sig_hist, QUANTILES)                                   # thresholds = quantiles of |signal| on the history
    hist_m = FR.run_rules(grid, MK(f, "hist"), NET)
    eval_m = FR.run_rules(grid, MK(f, "test"), NET)
    i = WF.select_rule(hist_m, MIN_TRADES)
    g = grid[i]
    RULES.append(g)
    BT_RULE.append(run_policy(FR.rule_policy([g], NET, "cpu"), MK(f, "test"), NET))
    BT_BH.append(run_policy(buy_and_hold_policy, MK(f, "test"), NET))
    m, bh = compute_metrics(BT_RULE[-1], PPY), compute_metrics(BT_BH[-1], PPY)
    assert np.isclose(m["sharpe"], eval_m["sharpe"][i], atol=1e-6)
    d = pd.DataFrame(grid).assign(fold=f + 1, **{f"hist_{k}": hist_m[k] for k in ("sharpe", "num_trades")},
                                  **{f"judged_{k}": eval_m[k] for k in ("sharpe", "cum_return_pct", "max_drawdown_pct", "num_trades")})
    all_rules.append(d)
    rule_rows.append({"fold": f + 1, "judged": ev_w, "rule": f"{g['signal']} q{g['q']} {g['mode']} hold>={g['min_hold']}",
                      "threshold": round(g["k"], 4), "history Sharpe": hist_m["sharpe"][i],
                      "Sharpe": m["sharpe"], "return %": m["cum_return_pct"], "max DD %": m["max_drawdown_pct"],
                      "trades": m["num_trades"], "in market %": m["exposure_pct"],
                      "B&H Sharpe": bh["sharpe"], "B&H return %": bh["cum_return_pct"],
                      "rules > 0 on judged": f"{(eval_m['sharpe'] > 0).sum()}/{len(grid)}",
                      "hist-judged rank corr": pd.Series(hist_m["sharpe"]).corr(pd.Series(eval_m["sharpe"]), method="spearman")})
rule_df = pd.DataFrame(rule_rows)
pd.concat(all_rules).to_csv(OUT / "rules_all_folds.csv", index=False)
rule_df.to_csv(OUT / "benchmark_rule.csv", index=False)
st_rule, st_bh = compute_metrics(WF.stitch(BT_RULE), PPY), compute_metrics(WF.stitch(BT_BH), PPY)
print("benchmark rule per fold (judged window, net of costs):")
print(rule_df.round(3).to_string(index=False))
print(f"\nall 4 judged windows joined: rule Sharpe {st_rule['sharpe']:+.2f}, return {st_rule['cum_return_pct']:+.1f} %, "
      f"max DD {st_rule['max_drawdown_pct']:.1f} %, {st_rule['num_trades']} trades | buy and hold Sharpe {st_bh['sharpe']:+.2f}, "
      f"return {st_bh['cum_return_pct']:+.1f} %, max DD {st_bh['max_drawdown_pct']:.1f} %")

## 2 · Gated and ungated Double DQN, 4 folds x 3 seeds (about 20-25 min per run; resumes)

In [ ]:
def bh_sharpe(f, split):
    return compute_metrics(run_policy(buy_and_hold_policy, MK(f, split), NET), PPY)["sharpe"]

def spec(f, seed, gated=True):
    env = EnvConfig(state="A", **ENV_OVER, **(WF.gate_config(RULES[f]) if gated else {}))
    return RunSpec(f"20c_wf_fold{f + 1}", "ddqn", seed=seed, env=env, agent_hp=AGENT_HP, total_steps=TOTAL_STEPS,
                   eval_every=EVAL_EVERY, min_val_trades=MIN_TRADES, fallback="cash",
                   fallback_min_score=max(0.0, bh_sharpe(f, "val")), tag="gated" if gated else "free")

VARIANTS = (True, False) if RUN_UNGATED else (True,)
t0 = time.time()
for f in range(len(WF.FOLDS)):
    print(f"--- fold {f + 1}: train {WF.FOLDS[f][0]}, checkpoint {WF.FOLDS[f][1]}, judged {WF.FOLDS[f][2]} | gate {WF.gate_config(RULES[f])}")
    for seed in SEEDS:
        for gated in VARIANTS:
            run(spec(f, seed, gated), FOLD_DIRS[f], FOLD_DIRS[f], OUT, eval_splits=("val", "test"))
print(f"agents done ({(time.time() - t0) / 60:.0f} min this session)")

## 3 · Results on the judged windows

In [ ]:
agent_rows, BT_AG = [], {}
for f in range(len(WF.FOLDS)):
    for seed in SEEDS:
        for gated in VARIANTS:
            sp = spec(f, seed, gated)
            rd = OUT / "models" / sp.run_id
            row = json.loads((rd / "metrics.json").read_text())["row"]
            mets, bts = backtest_saved(rd, ["test"], prepared_dir=FOLD_DIRS[f], factor_dir=FOLD_DIRS[f], device="cpu")
            assert np.isclose(mets["test"]["sharpe"], row["test_sharpe"], atol=1e-6)
            BT_AG[(f, seed, gated)] = bts["test"]
            agent_rows.append({"fold": f + 1, "seed": seed, "agent": "gated" if gated else "ungated",
                               "stayed flat": row["fallback_used"], "ckpt Sharpe (checkpoint window)": row["ckpt_val_sharpe"],
                               "Sharpe": row["test_sharpe"], "return %": row["test_cum_return_pct"],
                               "max DD %": row["test_max_drawdown_pct"], "trades": row["test_num_trades"],
                               "in market %": row["test_exposure_pct"]})
ag = pd.DataFrame(agent_rows)
ag.to_csv(OUT / "agents_judged.csv", index=False)
print("every agent run on its judged window (stayed flat = could not beat cash and B&H on its checkpoint window):")
print(ag.round(3).to_string(index=False))

def ms(x, fmt="+.2f"):
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    return f"{x.mean():{fmt}} ± {x.std(ddof=1):.2f}" if len(x) > 1 else f"{x.mean():{fmt}}"
summary = []
for f in range(len(WF.FOLDS)):
    r = rule_df.iloc[f]
    summary.append({"fold": f + 1, "judged": WF.FOLDS[f][2], "policy": "benchmark rule", "Sharpe": f"{r['Sharpe']:+.2f}",
                    "return %": f"{r['return %']:+.1f}", "max DD %": f"{r['max DD %']:.1f}", "trades": int(r["trades"])})
    for v in ag.agent.unique():
        g = ag[(ag.fold == f + 1) & (ag.agent == v)]
        summary.append({"fold": f + 1, "judged": WF.FOLDS[f][2], "policy": f"{v} agent ({len(g)} seeds)", "Sharpe": ms(g.Sharpe),
                        "return %": ms(g["return %"], "+.1f"), "max DD %": ms(g["max DD %"], ".1f"), "trades": ms(g.trades, ".0f")})
    summary.append({"fold": f + 1, "judged": WF.FOLDS[f][2], "policy": "buy and hold", "Sharpe": f"{r['B&H Sharpe']:+.2f}",
                    "return %": f"{r['B&H return %']:+.1f}", "max DD %": f"{compute_metrics(BT_BH[f], PPY)['max_drawdown_pct']:.1f}", "trades": 1})
joined = [{"policy": "benchmark rule", **{k: st_rule[k] for k in ("sharpe", "cum_return_pct", "max_drawdown_pct", "num_trades", "exposure_pct")}},
          {"policy": "buy and hold", **{k: st_bh[k] for k in ("sharpe", "cum_return_pct", "max_drawdown_pct", "num_trades", "exposure_pct")}},
          {"policy": "cash", "sharpe": 0.0, "cum_return_pct": 0.0, "max_drawdown_pct": 0.0, "num_trades": 0, "exposure_pct": 0.0}]
for v in VARIANTS:
    per_seed = [compute_metrics(WF.stitch([BT_AG[(f, s, v)] for f in range(len(WF.FOLDS))]), PPY) for s in SEEDS]
    for s, m in zip(SEEDS, per_seed):
        joined.append({"policy": f"{'gated' if v else 'ungated'} agent, seed {s}", **{k: m[k] for k in joined[0] if k != "policy"}})
joined = pd.DataFrame(joined)
summary = pd.DataFrame(summary)
print("\nper fold (judged window, net of costs; agents mean ± std over seeds):")
print(summary.to_string(index=False))
print("\nall 4 judged windows joined (about 11 months out of sample):")
print(joined.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
st = WF.stitch(BT_RULE); ax.plot(R.as_datetime(st["time_ns"]), st["equity"][1:], lw=1.6, color="C0", label="benchmark rule")
st = WF.stitch(BT_BH); ax.plot(R.as_datetime(st["time_ns"]), st["equity"][1:], lw=1.2, ls="--", color="black", label="buy and hold")
for k, v in enumerate(VARIANTS):
    for s in SEEDS:
        st = WF.stitch([BT_AG[(f, s, v)] for f in range(len(WF.FOLDS))])
        ax.plot(R.as_datetime(st["time_ns"]), st["equity"][1:], lw=0.9, alpha=0.8, color=f"C{1 + k}",
                label=f"{'gated' if v else 'ungated'} agent" if s == SEEDS[0] else None)
for f in range(1, len(WF.FOLDS)):
    ax.axvline(R.as_datetime(BT_RULE[f]["time_ns"][:1])[0], color="grey", lw=0.6, ls=":")
ax.set_title("Judged windows T3, T4, V1, V2 joined (each starts flat), net of costs", fontsize=10)
ax.set_ylabel("equity (start = 1)"); ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.tick_params(axis="x", rotation=30)
fig.tight_layout(); fig.savefig(REP / "e20c_walkforward_equity.png", dpi=130); plt.show()

In [ ]:
(REP / "e20c_walkforward.md").write_text(
    "# E20c: walk-forward, looser gate (rule thresholds q 0.90 / 0.95), gated vs ungated Double DQN (train + validation only)\n\n"
    f"Factors: {F} (train cross-fitted, validation E15). Agent: E16 hyperparameters, {len(SEEDS)} seeds x {TOTAL_STEPS:,} steps, "
    f"checkpoint on the window before the judged one (>= {MIN_TRADES} trades), stays flat unless it beats cash and buy and hold there.\n\n"
    "## Benchmark rule per fold\n\n" + rule_df.round(3).to_markdown(index=False)
    + "\n\n## Per fold\n\n" + summary.to_markdown(index=False)
    + "\n\n## All judged windows joined\n\n" + joined.round(3).to_markdown(index=False)
    + "\n\n## Every agent run\n\n" + ag.round(3).to_markdown(index=False)
    + "\n\nFigure: e20c_walkforward_equity.png\n", encoding="utf-8")
print("wrote", REP / "e20c_walkforward.md", "|", OUT)

In [ ]:
AUTO_DISCONNECT = True               # release the GPU once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()